# Módulo 08 · Series temporales — Soluciones de los ejercicios

Este notebook es **autocontenido**: reproduce la preparación mínima del notebook del módulo (datos de CO₂ de Mauna Loa, partición temporal, modelos de referencia y el esquema de *backtesting*) y resuelve los cuatro ejercicios finales.

| Ejercicio | Qué se pide |
|---|---|
| 1 | SARIMA con toda la serie, pronóstico a 24 meses con intervalo y 3 frases para un responsable político |
| 2 | Variantes de Holt-Winters (estacionalidad multiplicativa, tendencia amortiguada) en el *backtesting* |
| 3 | Búsqueda de SARIMA por AIC y comparación con el MAE de *backtesting* |
| 4 | Flujo completo sobre AirPassengers (o datos de tu empresa) |

> Nota de rendimiento: fijamos un solo hilo para las librerías numéricas. Ajustar decenas de SARIMA en paralelo con varios hilos por modelo suele ir **más lento**, no más rápido.

In [ ]:
import os
for var in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ.setdefault(var, "1")

import time
import itertools
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

## Preparación (igual que en el notebook del módulo)

In [ ]:
co2 = sm.datasets.co2.load_pandas().data["co2"]
serie = co2.resample("MS").mean().interpolate()
serie.index.freq = "MS"
print(f"{len(serie)} meses: {serie.index[0]:%Y-%m} → {serie.index[-1]:%Y-%m}")

H = 24
train, test = serie[:-H], serie[-H:]

# Orígenes del backtesting del notebook: diciembre de 1994 a diciembre de 2000, horizonte 12 meses
ORIGENES = pd.date_range("1994-12-01", "2000-12-01", freq="12MS")


def cortes(s, origenes, h=12):
    """Genera (año, entrenamiento, prueba) para un backtesting de origen móvil."""
    for corte in origenes:
        tr = s[:corte]
        te = s[corte + pd.DateOffset(months=1): corte + pd.DateOffset(months=h)]
        yield corte.year, tr, te


def mae(real, pred):
    return float(np.mean(np.abs(np.asarray(real) - np.asarray(pred))))


def mape(real, pred):
    real, pred = np.asarray(real), np.asarray(pred)
    return float(np.mean(np.abs((real - pred) / real)) * 100)

---
## Ejercicio 1 · SARIMA con toda la serie y pronóstico a 24 meses

Reentrenamos el mismo $\text{SARIMA}(1,1,1)(0,1,1)_{12}$ del notebook, ahora con **todos** los datos (hasta diciembre de 2001), y pronosticamos enero 2002 – diciembre 2003. Para poder decir "con qué error" medimos además, con *backtesting* de origen móvil, el error esperado **a 24 meses vista** (el del notebook era a 12 meses).

In [ ]:
sarima_total = SARIMAX(serie, order=(1, 1, 1), seasonal_order=(0, 1, 1, 12)).fit(disp=False)
fc_total = sarima_total.get_forecast(24)
pron = fc_total.predicted_mean
ic95 = fc_total.conf_int(alpha=0.05)
ic80 = fc_total.conf_int(alpha=0.20)

tabla_pron = pd.DataFrame({"pronóstico": pron, "IC95 inf": ic95.iloc[:, 0], "IC95 sup": ic95.iloc[:, 1]})
tabla_pron["ancho IC95"] = tabla_pron["IC95 sup"] - tabla_pron["IC95 inf"]
print(tabla_pron.iloc[[0, 5, 11, 17, 23]].round(2))

media_2001 = serie["2001"].mean()
media_2002, media_2003 = pron["2002"].mean(), pron["2003"].mean()
print(f"\nMedia anual 2001 (real):       {media_2001:.2f} ppm")
print(f"Media anual 2002 (pronóstico): {media_2002:.2f} ppm  (+{media_2002 - media_2001:.2f})")
print(f"Media anual 2003 (pronóstico): {media_2003:.2f} ppm  (+{media_2003 - media_2002:.2f})")
print(f"Máximo pronosticado: {pron.max():.2f} ppm en {pron.idxmax():%Y-%m}; mínimo: {pron.min():.2f} ppm en {pron.idxmin():%Y-%m}")

In [ ]:
plt.figure(figsize=(12, 4))
plt.plot(serie["1996":], "k", label="histórico")
plt.plot(pron.index, pron, color="C3", label="pronóstico SARIMA")
plt.fill_between(pron.index, ic95.iloc[:, 0], ic95.iloc[:, 1], color="C3", alpha=0.15, label="IC 95 %")
plt.fill_between(pron.index, ic80.iloc[:, 0], ic80.iloc[:, 1], color="C3", alpha=0.25, label="IC 80 %")
plt.axvline(serie.index[-1], color="gray", ls=":")
plt.title("CO₂ en Mauna Loa: pronóstico a 24 meses (enero 2002 – diciembre 2003)")
plt.ylabel("ppm"); plt.legend(loc="upper left"); plt.show()

**¿Qué error podemos esperar a 24 meses?** Repetimos el *backtesting* con horizonte 24 (orígenes de 1993 a 1999 para que los 24 meses de prueba existan) y lo comparamos con el ingenuo estacional y con la referencia "estacional + deriva".

In [ ]:
filas = []
for anio, tr, te in cortes(serie, pd.date_range("1993-12-01", "1999-12-01", freq="12MS"), h=24):
    p_sa = SARIMAX(tr, order=(1, 1, 1), seasonal_order=(0, 1, 1, 12)).fit(disp=False).forecast(24)
    p_ie = np.tile(tr.iloc[-12:].values, 2)
    subida = (tr.iloc[-12:].values - tr.iloc[-24:-12].values).mean()
    p_ed = np.concatenate([tr.iloc[-12:].values + subida * (a + 1) for a in range(2)])
    filas.append({"origen": anio, "SARIMA": mae(te, p_sa), "Ingenuo estacional": mae(te, p_ie),
                  "Estacional + deriva": mae(te, p_ed), "SARIMA MAPE (%)": mape(te, p_sa)})
bt24 = pd.DataFrame(filas).set_index("origen")
print(bt24.round(3))
print("\nMedia:\n", bt24.mean().round(3), sep="")
mejora_ie = 1 - bt24["SARIMA"].mean() / bt24["Ingenuo estacional"].mean()
mejora_ed = 1 - bt24["SARIMA"].mean() / bt24["Estacional + deriva"].mean()
print(f"\nReducción del MAE frente al ingenuo estacional: {mejora_ie:.0%}")
print(f"Reducción del MAE frente a estacional + deriva:  {mejora_ed:.0%}")

**Interpretación y mensaje para un responsable político** (las cifras proceden de las salidas anteriores):

1. *Qué esperar:* la concentración de CO₂ seguirá subiendo al ritmo de los últimos años, unas **1,6–1,7 ppm al año**: de una media de 370,9 ppm en 2001 pasaremos a unas 372,6 ppm en 2002 y unas 374,1 ppm en 2003, con el pico estacional en mayo (unas 377 ppm en mayo de 2003) y el valle en septiembre-octubre, como cada año.
2. *Con qué incertidumbre:* el intervalo del 95 % es estrecho al principio (±0,6 ppm el primer mes) y se ensancha hasta ±1,8 ppm a dos años vista; cuando repetimos este ejercicio en el pasado, el modelo se equivocó de media **0,66 ppm a 24 meses** (un 0,2 %).
3. *Cuánto aporta el modelo:* ese error es un **75 % menor** que el de la regla "repetir lo del año pasado" (2,6 ppm) y un 44 % menor que el de "repetir lo del año pasado sumando la subida media anual" (1,2 ppm). En cualquier caso, la conclusión no depende del modelo: el CO₂ sigue creciendo de forma sostenida y nada indica que vaya a estabilizarse a corto plazo.

Observa que, a 24 meses, el error del SARIMA (0,66 ppm) es mayor que el de 12 meses que vimos en el notebook (0,44 ppm): el error esperado depende del horizonte y hay que comunicarlo para el horizonte que se va a usar.

---
## Ejercicio 2 · Variantes de Holt-Winters

Comparamos en el mismo *backtesting* del notebook (7 orígenes, horizonte 12) cuatro configuraciones:

- `add/add`: la del notebook (tendencia y estacionalidad aditivas).
- `add/mul`: estacionalidad **multiplicativa** (la amplitud estacional crece con el nivel).
- `add amort./add`: tendencia **amortiguada** (`damped_trend=True`), que aplana la pendiente con el horizonte.
- `add amort./mul`: ambas cosas a la vez.

In [ ]:
variantes = {
    "add/add":        dict(trend="add", seasonal="add"),
    "add/mul":        dict(trend="add", seasonal="mul"),
    "add amort./add": dict(trend="add", seasonal="add", damped_trend=True),
    "add amort./mul": dict(trend="add", seasonal="mul", damped_trend=True),
}
filas = []
for anio, tr, te in cortes(serie, ORIGENES):
    fila = {"origen": anio}
    for nombre, kw in variantes.items():
        fila[nombre] = mae(te, ExponentialSmoothing(tr, seasonal_periods=12, **kw).fit().forecast(12))
    filas.append(fila)
bt_hw = pd.DataFrame(filas).set_index("origen")
print(bt_hw.round(3))
print("\nMAE medio (ppm):\n", bt_hw.mean().sort_values().round(3), sep="")

In [ ]:
hw_amort = ExponentialSmoothing(train, trend="add", seasonal="add", damped_trend=True, seasonal_periods=12).fit()
print(f"Parámetro de amortiguación φ estimado: {hw_amort.params['damping_trend']:.3f}")
res_test = {nombre: mae(test, ExponentialSmoothing(train, seasonal_periods=12, **kw).fit().forecast(H))
            for nombre, kw in variantes.items()}
print("\nMAE en el test de 24 meses del notebook:")
print(pd.Series(res_test).sort_values().round(3))

**Interpretación.**

- **Estacionalidad multiplicativa:** en el *backtesting* queda empatada con la aditiva (0,456 frente a 0,454 ppm de MAE medio) y en el test de 24 meses es peor (0,446 frente a 0,236). Tiene sentido: el CO₂ solo sube un ~17 % entre 1958 y 2001 (de 316 a 371 ppm), así que la amplitud del ciclo anual apenas cambia con el nivel y no hay nada "multiplicativo" que capturar. Con series cuya oscilación crece con el nivel (ventas de una empresa que se multiplica por 5, o AirPassengers en el ejercicio 4) la multiplicativa sí marca diferencias.
- **Tendencia amortiguada:** mejora **muy ligeramente** (0,445 frente a 0,454 ppm en el *backtesting*; 0,209 frente a 0,236 en el test). Pero el $\phi$ estimado es 0,995, es decir, casi no amortigua: el modelo "decide" que la tendencia continúa. La diferencia (≈ 0,01 ppm) es mucho menor que la variación entre orígenes (de 0,18 a 1,15 ppm), así que no es una mejora relevante.
- **Conclusión:** ninguna variante mejora de forma apreciable la configuración `add/add` del notebook. Podríamos adoptar la amortiguada (es más prudente si la tendencia llegara a frenarse y aquí no penaliza), pero sería una decisión de criterio, no de evidencia. Lección general: probar variantes es barato, pero solo se adoptan si mejoran el *backtesting* de forma clara, no el ajuste en entrenamiento.

---
## Ejercicio 3 · Búsqueda de SARIMA por AIC

Fijamos $d = 1$ y $D = 1$ (lo que indicó la prueba ADF en el notebook) y recorremos $(p, q) \in \{0,1,2\}^2$ y $(P, Q) \in \{0,1\}^2$: **36 modelos**. El AIC se calcula sobre el conjunto de entrenamiento del notebook (hasta diciembre de 1999).

Para mantener el tiempo de ejecución acotado, el *backtesting* (7 orígenes × 12 meses) solo se calcula para los **8 mejores por AIC** y para el modelo del notebook, en lugar de para los 36 (serían 252 ajustes).

In [ ]:
t0 = time.time()
resultados = []
for p, q, P, Q in itertools.product([0, 1, 2], [0, 1, 2], [0, 1], [0, 1]):
    try:
        m = SARIMAX(train, order=(p, 1, q), seasonal_order=(P, 1, Q, 12)).fit(disp=False)
        resultados.append({"modelo": f"({p},1,{q})({P},1,{Q})12", "orden": (p, 1, q), "estacional": (P, 1, Q, 12),
                           "AIC": m.aic, "MAE test 24m": mae(test, m.forecast(H))})
    except Exception as e:  # algún ajuste puede no converger
        print(f"({p},1,{q})({P},1,{Q}) falló: {e}")
busqueda = pd.DataFrame(resultados).sort_values("AIC").reset_index(drop=True)
print(f"{len(busqueda)} modelos ajustados en {time.time() - t0:.0f} s\n")
print(busqueda[["modelo", "AIC", "MAE test 24m"]].head(10).round(3).to_string())
print("...\nPeores por AIC:")
print(busqueda[["modelo", "AIC", "MAE test 24m"]].tail(3).round(3).to_string())

In [ ]:
candidatos = busqueda.head(8).copy()
if "(1,1,1)(0,1,1)12" not in set(candidatos["modelo"]):
    candidatos = pd.concat([candidatos, busqueda[busqueda["modelo"] == "(1,1,1)(0,1,1)12"]])

t0 = time.time()
mae_bt = {}
for _, fila in candidatos.iterrows():
    errores = [mae(te, SARIMAX(tr, order=fila["orden"], seasonal_order=fila["estacional"]).fit(disp=False).forecast(12))
               for _, tr, te in cortes(serie, ORIGENES)]
    mae_bt[fila["modelo"]] = np.mean(errores)
candidatos["MAE backtesting"] = candidatos["modelo"].map(mae_bt)
candidatos["rango AIC"] = candidatos.index + 1
candidatos["rango MAE bt"] = candidatos["MAE backtesting"].rank().astype(int)
print(f"Backtesting de {len(candidatos)} modelos en {time.time() - t0:.0f} s\n")
print(candidatos[["modelo", "AIC", "rango AIC", "MAE backtesting", "rango MAE bt", "MAE test 24m"]].round(3).to_string(index=False))

mejor_aic = candidatos.iloc[0]["modelo"]
mejor_bt = candidatos.sort_values("MAE backtesting").iloc[0]["modelo"]
print(f"\nMejor AIC: {mejor_aic}   ·   Mejor MAE de backtesting: {mejor_bt}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(candidatos["AIC"], candidatos["MAE backtesting"])
for _, f in candidatos.iterrows():
    ax.annotate(f["modelo"], (f["AIC"], f["MAE backtesting"]), fontsize=8, xytext=(4, 2), textcoords="offset points")
ax.set_xlabel("AIC (entrenamiento hasta 1999)"); ax.set_ylabel("MAE medio backtesting (ppm)")
ax.set_title("¿Menor AIC = menor error de pronóstico?"); plt.show()

**Interpretación.**

- Los modelos **sin componentes estacionales** ($P = Q = 0$) son con diferencia los peores por AIC (≈ 486–528 frente a ≈ 251), y los 10 mejores incluyen todos $Q = 1$: la ACF de la serie diferenciada mostraba un pico fuerte en el retardo 12, que es justo lo que captura el término MA estacional.
- **El mejor AIC no coincide con el mejor MAE de *backtesting*.** El $(1,1,1)(0,1,1)_{12}$ del notebook tiene el mejor AIC (251,2) pero queda 7.º de 8 en *backtesting* (0,443 ppm); el mejor en *backtesting* es el más sencillo, el "modelo de las aerolíneas" $(0,1,1)(0,1,1)_{12}$ (0,416 ppm), que es 7.º por AIC. Y en el test de 24 meses el mejor de esta lista es otro, el $(1,1,1)(1,1,1)_{12}$ (0,334 ppm).
- Las diferencias de MAE entre candidatos son pequeñas (0,42–0,44 ppm, menos de 0,03 ppm), mucho menores que la variación entre orígenes: cualquiera de ellos es razonable.
- **Por qué no coinciden:** el AIC mide el ajuste *un paso adelante* dentro de la muestra, penalizado por el número de parámetros; el *backtesting* mide el error *a 1–12 meses* fuera de la muestra, que es lo que nos interesa. Son criterios relacionados, pero distintos.
- **Decisión práctica:** usar el AIC para **filtrar** candidatos razonables (descartar los que no tienen parte estacional) y el *backtesting* para **elegir**, prefiriendo el modelo más sencillo cuando las diferencias son pequeñas. Aquí, el $(0,1,1)(0,1,1)_{12}$ gana en *backtesting* y además tiene solo 2 parámetros.

---
## Ejercicio 4 · Flujo completo con AirPassengers

Pasajeros mensuales de líneas aéreas internacionales (miles), 1949–1960. Es la serie clásica de Box y Jenkins: tendencia creciente y estacionalidad **cuya amplitud crece con el nivel** (multiplicativa).

La descarga necesita internet. Si falla, usamos una copia de los 144 valores incluida aquí (plan B offline), así el notebook se ejecuta siempre. Para aplicar el flujo a datos de tu empresa, basta con sustituir `ap` por una `pd.Series` mensual con índice de fechas y `freq="MS"`.

In [ ]:
VALORES_AIRPASSENGERS = [
    112, 118, 132, 129, 121, 135, 148, 148, 136, 119, 104, 118, 115, 126, 141, 135, 125, 149, 170, 170, 158, 133, 114, 140,
    145, 150, 178, 163, 172, 178, 199, 199, 184, 162, 146, 166, 171, 180, 193, 181, 183, 218, 230, 242, 209, 191, 172, 194,
    196, 196, 236, 235, 229, 243, 264, 272, 237, 211, 180, 201, 204, 188, 235, 227, 234, 264, 302, 293, 259, 229, 203, 229,
    242, 233, 267, 269, 270, 315, 364, 347, 312, 274, 237, 278, 284, 277, 317, 313, 318, 374, 413, 405, 355, 306, 271, 306,
    315, 301, 356, 348, 355, 422, 465, 467, 404, 347, 305, 336, 340, 318, 362, 348, 363, 435, 491, 505, 404, 359, 310, 337,
    360, 342, 406, 396, 420, 472, 548, 559, 463, 407, 362, 405, 417, 391, 419, 461, 472, 535, 622, 606, 508, 461, 390, 432,
]
indice_ap = pd.date_range("1949-01-01", periods=144, freq="MS")
try:
    datos = sm.datasets.get_rdataset("AirPassengers").data
    ap = pd.Series(datos["value"].values.astype(float), index=indice_ap, name="pasajeros")
    print("AirPassengers descargado de Rdatasets.")
except Exception as e:
    print(f"No se pudo descargar AirPassengers ({type(e).__name__}). Usamos la copia local incluida en el notebook.")
    ap = pd.Series(np.array(VALORES_AIRPASSENGERS, dtype=float), index=indice_ap, name="pasajeros")
ap.index.freq = "MS"
assert len(ap) == 144
print(f"{len(ap)} meses: {ap.index[0]:%Y-%m} → {ap.index[-1]:%Y-%m}; de {ap.iloc[0]:.0f} a {ap.iloc[-1]:.0f} miles de pasajeros")

### 4.1 Exploración y descomposición
La oscilación anual crece con el nivel, así que aplicamos **logaritmo**: convierte la estacionalidad multiplicativa en aditiva y estabiliza la varianza. STL se aplica a la serie en logaritmos.

In [ ]:
amplitud = ap.groupby(ap.index.year).agg(lambda x: x.max() - x.min())
print("Amplitud estacional (máx − mín del año), primeros y últimos años:")
print(amplitud.iloc[[0, 1, -2, -1]])

log_ap = np.log(ap)
fig, axes = plt.subplots(1, 2, figsize=(13, 3.5))
ap.plot(ax=axes[0], title="AirPassengers (miles)")
log_ap.plot(ax=axes[1], title="log(AirPassengers)"); plt.tight_layout(); plt.show()

stl_ap = STL(log_ap, period=12).fit()
stl_ap.plot().set_size_inches(12, 7); plt.show()
var_res = stl_ap.resid.var()
fuerza_tend = max(0, 1 - var_res / (stl_ap.trend + stl_ap.resid).var())
fuerza_est = max(0, 1 - var_res / (stl_ap.seasonal + stl_ap.resid).var())
print(f"Fuerza de la tendencia: {fuerza_tend:.2f} · fuerza de la estacionalidad: {fuerza_est:.2f}")

In [ ]:
for nombre, s in [("log original", log_ap), ("1ª diferencia", log_ap.diff()), ("1ª dif. + dif. estacional", log_ap.diff().diff(12))]:
    print(f"{nombre:28s} p-valor ADF = {adfuller(s.dropna())[1]:.4f}")
fig, axes = plt.subplots(1, 2, figsize=(13, 3.5))
plot_acf(log_ap.diff().diff(12).dropna(), lags=36, ax=axes[0])
plot_pacf(log_ap.diff().diff(12).dropna(), lags=36, ax=axes[1])
plt.tight_layout(); plt.show()

### 4.2 Partición, referencias y modelos
Reservamos los últimos 24 meses (1959–1960) como test. Modelos:

- Referencias: ingenuo, ingenuo estacional, deriva y estacional + deriva (como en el notebook).
- Holt-Winters aditivo y multiplicativo sobre la serie original.
- SARIMA $(0,1,1)(0,1,1)_{12}$ sobre el **logaritmo** (el "modelo de las aerolíneas" de Box y Jenkins); el pronóstico se devuelve a la escala original con la exponencial.

In [ ]:
H_AP = 24
tr_ap, te_ap = ap[:-H_AP], ap[-H_AP:]


def referencias(tr, h):
    """Las cuatro referencias ingenuas del notebook para un horizonte h múltiplo de 12."""
    n_anios = h // 12
    sub = (tr.iloc[-12:].values - tr.iloc[-24:-12].values).mean()
    return {
        "Ingenuo": np.repeat(tr.iloc[-1], h),
        "Ingenuo estacional": np.tile(tr.iloc[-12:].values, n_anios),
        "Deriva": tr.iloc[-1] + (tr.iloc[-1] - tr.iloc[0]) / (len(tr) - 1) * np.arange(1, h + 1),
        "Estacional + deriva": np.concatenate([tr.iloc[-12:].values + sub * (a + 1) for a in range(n_anios)]),
    }


def modelos_ap(tr, h):
    """Pronósticos de los modelos candidatos para AirPassengers."""
    pred = referencias(tr, h)
    pred["HW aditivo"] = ExponentialSmoothing(tr, trend="add", seasonal="add", seasonal_periods=12).fit().forecast(h).values
    pred["HW multiplicativo"] = ExponentialSmoothing(tr, trend="add", seasonal="mul", seasonal_periods=12).fit().forecast(h).values
    sar = SARIMAX(np.log(tr), order=(0, 1, 1), seasonal_order=(0, 1, 1, 12)).fit(disp=False)
    pred["SARIMA log"] = np.exp(sar.forecast(h).values)
    return pred


pred_ap = modelos_ap(tr_ap, H_AP)
comparacion_ap = pd.DataFrame({"MAE (miles)": {k: mae(te_ap, v) for k, v in pred_ap.items()},
                               "MAPE (%)": {k: mape(te_ap, v) for k, v in pred_ap.items()}}).sort_values("MAE (miles)")
print(comparacion_ap.round(2))

In [ ]:
sar_ap = SARIMAX(np.log(tr_ap), order=(0, 1, 1), seasonal_order=(0, 1, 1, 12)).fit(disp=False)
print(sar_ap.summary().tables[1])
fc_ap = sar_ap.get_forecast(H_AP)
ic_ap = np.exp(fc_ap.conf_int(alpha=0.05))

plt.figure(figsize=(12, 4))
plt.plot(ap, "k", label="real")
plt.plot(te_ap.index, pred_ap["Ingenuo estacional"], "--", label="ingenuo estacional")
plt.plot(te_ap.index, pred_ap["HW aditivo"], label="HW aditivo")
plt.plot(te_ap.index, pred_ap["HW multiplicativo"], label="HW multiplicativo")
plt.plot(te_ap.index, pred_ap["SARIMA log"], label="SARIMA (log)")
plt.fill_between(te_ap.index, ic_ap.iloc[:, 0], ic_ap.iloc[:, 1], alpha=0.2, label="IC 95 % SARIMA")
plt.axvline(te_ap.index[0], color="gray", ls=":"); plt.legend(loc="upper left")
plt.title("AirPassengers: pronóstico de 1959–1960"); plt.show()

### 4.3 Backtesting de origen móvil
Orígenes en diciembre de 1954 a 1959, horizonte 12 meses (6 cortes). Mismo esquema que en el notebook.

In [ ]:
filas = []
for anio, tr, te in cortes(ap, pd.date_range("1954-12-01", "1959-12-01", freq="12MS")):
    if len(te) < 12:
        continue
    p = modelos_ap(tr, 12)
    filas.append({"origen": anio, **{k: mape(te, v) for k, v in p.items() if k != "Deriva"}})
bt_ap = pd.DataFrame(filas).set_index("origen")
print("MAPE (%) por origen:")
print(bt_ap.round(2))
print("\nMAPE medio (%):\n", bt_ap.mean().sort_values().round(2), sep="")

In [ ]:
sar_ap.plot_diagnostics(figsize=(12, 6)); plt.tight_layout(); plt.show()
from statsmodels.stats.diagnostic import acorr_ljungbox
print(acorr_ljungbox(sar_ap.resid[13:], lags=[12, 24]).round(3))

**Interpretación del ejercicio 4.**

- **Descomposición:** tendencia muy fuerte y estacionalidad marcada (picos en julio-agosto, valles en noviembre); la amplitud del ciclo anual pasa de 44 mil pasajeros en 1949 a 232 mil en 1960. Por eso trabajamos en logaritmos (SARIMA) o con estacionalidad multiplicativa (Holt-Winters).
- **Estacionariedad:** la serie en logaritmos no es estacionaria (p-valor ADF 0,42); con la primera diferencia queda en el límite (0,07) y con diferencia regular + estacional sí lo es (0,0002). La ACF de la serie doblemente diferenciada muestra picos en los retardos 1 y 12, coherentes con el modelo $(0,1,1)(0,1,1)_{12}$.
- **Referencias:** el ingenuo estacional se equivoca un 15,5 % en 1959–1960 y un 10,7 % de media en el *backtesting*, porque ignora el crecimiento (~10–15 % anual). La referencia estacional + deriva lo corrige en gran parte (11,3 % y 5,7 %): es la referencia exigente que hay que batir.
- **Modelos:** en el *backtesting* (6 orígenes, más fiable que una sola partición) el mejor es **Holt-Winters multiplicativo** (MAPE medio 3,9 %), seguido de SARIMA sobre el logaritmo (4,2 %); Holt-Winters aditivo (5,4 %) apenas mejora a la referencia estacional + deriva (5,7 %), porque no reproduce bien una oscilación que crece. En la partición única de 1959–1960 el orden cambia algo (HW multiplicativo 6,4 %, HW aditivo 6,6 %, SARIMA 8,5 %): otro ejemplo de por qué no conviene decidir con un único corte.
- **Residuos:** la prueba de Ljung-Box no rechaza que los residuos del SARIMA sean ruido blanco (p-valores 0,91 y 0,80 a 12 y 24 retardos), así que el modelo ha capturado la estructura principal.
- **Para tu empresa:** el mismo flujo (descomposición → referencias → modelos → *backtesting*) sirve tal cual. Dos preguntas clave: (a) ¿la oscilación crece con el nivel? → logaritmos o estacionalidad multiplicativa; (b) ¿el modelo bate con claridad a la referencia estacional + deriva en el *backtesting*? → si no, quédate con la referencia.